# Integrated Energy System Design

This notebook combines PV solar panels and heat pump into one view, showing how they interact over different time scales (hourly, monthly, seasonal). More energy systems can be added here later (battery, EV, household loads).

In [6]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))

from home_energy.domain import PVSegment, HeatPumpConfig
from home_energy.solar import (
    UTRECHT_SITE,
    build_hourly_figure as build_solar_hourly,
    simulate_pv_system,
)
from home_energy.heat_pump import (
    simulate_heat_pump,
)

In [7]:
# Define PV segments
pv_segments = [
    PVSegment(name='South roof', capacity_kwp=5.0, azimuth_deg=180.0, tilt_deg=35.0),
    PVSegment(name='East roof', capacity_kwp=3.0, azimuth_deg=90.0, tilt_deg=25.0),
]

# Define heat pump
hp_config = HeatPumpConfig(
    enabled=True,
    annual_heat_demand_kwh=8000,
    seasonal_cop=4.0,
)

print(f"PV total capacity: {sum(seg.capacity_kwp for seg in pv_segments)} kWp")
print(f"Heat pump: {hp_config.annual_heat_demand_kwh} kWh heat demand, COP {hp_config.seasonal_cop}")

PV total capacity: 8.0 kWp
Heat pump: 8000 kWh heat demand, COP 4.0


In [ ]:
# Run simulations
pv_result = simulate_pv_system(
    pv_segments,
    year=2026,
    site=UTRECHT_SITE,
    temperature_loss_percent=0.04,
)

hp_result = simulate_heat_pump(
    hp_config,
    year=2026,
)

print(f"PV annual output: {pv_result.total_annual_kwh:.0f} kWh")
print(f"Heat pump annual electricity: {hp_result.annual_electricity_kwh:.0f} kWh")

AttributeError: 'SolarSimulationResult' object has no attribute 'annual_kwh'

## Comparison: PV Production vs Heat Pump Consumption

The following charts show how solar production and heat pump electricity consumption interact across different time scales.

In [ ]:
import plotly.graph_objects as go

# Aggregate hourly data by hour-of-day
pv_by_hour = [0.0] * 24
hp_by_hour = [0.0] * 24
hour_counts = [0] * 24

for index, timestamp in enumerate(pv_result.timestamps):
    hour = timestamp.hour
    hour_counts[hour] += 1
    pv_by_hour[hour] += pv_result.total_hourly_kwh[index]
    if index < len(hp_result.hourly_electricity_kwh):
        hp_by_hour[hour] += hp_result.hourly_electricity_kwh[index]

# Average by hour count
for hour in range(24):
    if hour_counts[hour] > 0:
        pv_by_hour[hour] /= hour_counts[hour]
        hp_by_hour[hour] /= hour_counts[hour]

# Hourly comparison
hourly_figure = go.Figure()

# Add PV production as bars
hourly_figure.add_trace(
    go.Bar(
        x=list(range(24)),
        y=pv_by_hour,
        name="PV Production",
        marker_color="gold",
        yaxis="y1",
    )
)

# Add heat pump consumption as line
hourly_figure.add_trace(
    go.Scatter(
        x=list(range(24)),
        y=hp_by_hour,
        mode="lines+markers",
        name="Heat Pump Consumption",
        line=dict(color="red", width=3),
        yaxis="y2",
    )
)

hourly_figure.update_layout(
    title="Hourly: PV Production vs Heat Pump Consumption",
    xaxis_title="Hour of day (average)",
    yaxis=dict(title="PV Production (kWh)", side="left"),
    yaxis2=dict(title="Heat Pump (kWh)", side="right", overlaying="y"),
    hovermode="x unified",
    template="plotly_white",
)
hourly_figure

AttributeError: 'SolarSimulationResult' object has no attribute 'hourly_kwh'

In [ ]:
# Monthly comparison
monthly_figure = go.Figure()

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# Add PV monthly
monthly_figure.add_trace(
    go.Bar(
        x=months,
        y=pv_result.total_monthly_kwh,
        name="PV Production",
        marker_color="gold",
    )
)

# Add heat pump monthly
monthly_figure.add_trace(
    go.Bar(
        x=months,
        y=hp_result.monthly_electricity_kwh,
        name="Heat Pump Consumption",
        marker_color="red",
    )
)

monthly_figure.update_layout(
    title="Monthly: PV Production vs Heat Pump Consumption",
    barmode="group",
    xaxis_title="Month",
    yaxis_title="Energy (kWh)",
    template="plotly_white",
)
monthly_figure

AttributeError: 'SolarSimulationResult' object has no attribute 'monthly_kwh'

## Summary

- **PV Annual Production**: {:.0f} kWh
- **Heat Pump Annual Electricity**: {:.0f} kWh
- **Ratio**: {:.1f}% of heat pump consumption covered by average PV production

This integrated view helps understand how solar production and heat pump loads interact seasonally, guiding investment and control strategies.

Future versions will add battery storage and EV charging to this view.
,
,
,
100


,

: {
: {
: 
,
: 
,
: 

: {
: 
,
: 
3.13

: 4,
: 5